In [60]:
import scanpy as sc
import numpy as np
import pertpy as pt
from scanpy import AnnData
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
ad = sc.read_h5ad("raw/GSE210950_hscArray.h5ad")
adata = ad[ad.obs['Filtered']=='False'].copy()

In [4]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=50)
sc.pp.normalize_total(adata, target_sum=1e4)
adata.layers['raw'] = adata.X.copy()
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=5000)
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [9]:
sc.tl.leiden(adata, resolution=0.5, iterations=2)

In [5]:
adata

AnnData object with n_obs × n_vars = 176564 × 19964
    obs: 'Perturbation', 'LIBID', 'SAMID', 'Donor', 'Treatment', 'Filtered', 'n_genes', 'n_genes_by_counts'
    var: 'featureid', 'feature_types', 'genome', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'genome', 'modality', 'log1p', 'hvg', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw'
    obsp: 'distances', 'connectivities'

In [32]:
adata.obs['perturbation'] = adata.obs['Perturbation'].str.upper().str.replace('NTC', 'control')
adata.obs['batch'] = adata.obs[['LIBID', 'Donor']].agg('-'.join, axis=1)

In [ ]:
ms = pt.tl.Mixscape()
ms.perturbation_signature(
    adata,
    pert_key="perturbation",
    control="control",
    split_by="batch",
)
ms.mixscape(
    adata,
    labels="perturbation",
    control="control",
    layer='X_pert'
)

In [34]:
adata.write_h5ad('preprocessed.h5ad')

In [ ]:
clearning_cols = ['perturbation', 'Donor', 'mixscape_class_global']
split_df_full = adata.obs[clearning_cols].copy()
np.random.seed(42)
split_df_full['split'] = np.random.choice(['train', 'val', 'test'], size=len(split_df_full), p=[0.7, 0.1, 0.2])

In [56]:
split_df_full['Donor'].value_counts()

Donor
Donor4    114083
Donor1     62481
Name: count, dtype: int64

In [ ]:
split_df_subsplit = split_df_full.copy()
split_df_subsplit.loc[split_df_subsplit['Donor'] == "Donor1", 'split'] = 'test'
split_df_subsplit.rename(columns={'Donor': 'subsplit'}, inplace=True)
split_df_subsplit.query('mixscape_class_global != "NP"').drop(columns=list(set(clearning_cols) - set(['Donor']))).to_parquet('split_df_nonpc_subsplit.parquet')